In [142]:
import re
import pandas as pd
import os

In [143]:
def extract_conflicts_clingo(output):
    match = re.search(r"Conflicts\s*:\s*(\d+)", output)
    if match:
        return int(match.group(1))
    return None

def extract_conflicts_wasp(text):
    match = re.search(r"Learned clauses\s*:\s*(\d+)", text)
    return int(match.group(1)) if match else None

def extract_params(name: str):
    regex = r"instance_(?P<ngroups>\d+)-(?P<partsize>\d+)-(?P<b>\d+)_(?P<type>unsat|sat)"
    m = re.search(regex, name)
    ngroups = int(m.group("ngroups"))
    partsize = int(m.group("partsize"))
    b = int(m.group("b"))
    t = m.group("type")
    alpha = b * 0.01
    C1 = ngroups * partsize 
    C2 = ngroups * partsize * (partsize+1) / 2 
    # print(f"ngroups: {ngroups} partsize: {partsize} b: {b} type: {t} alpha: {alpha} C1: {C1} C2: {C2}")
    bound = round(alpha * C1) if t == "sat" else round(C1 + alpha * (C2 - C1)) 

    return {
        "ngroups": ngroups,
        "part_size": partsize,
        "bound": bound,
        "alpha": alpha,
        "type": t,
    }

In [144]:
extract_params("instance_10-100-15_unsat.asp.out")

{'ngroups': 10,
 'part_size': 100,
 'bound': 8425,
 'alpha': 0.15,
 'type': 'unsat'}

In [145]:
with open("outclingo/instance_10-1000-60_unsat.asp.out", "r") as f:
    template = f.read()
    print(extract_conflicts_clingo(template)) 

74180


In [146]:
with open("outwasp/instance_10-1000-60_unsat.asp.err", "r") as f:
    template = f.read()
    # print(template)
    print(extract_conflicts_wasp(template)) 

27791


In [147]:
df = pd.DataFrame(columns=["solver", "bound", "part_size", "conflicts"])

def create_df(df_input, solver):
    dirout = f"outclingo" if solver == "clingo" else "outwasp"
    for f in os.listdir(dirout):
        params = extract_params(f)
        path_file = f"{dirout}/{f}"
        with open(path_file, "r") as f:
            conflits = extract_conflicts_clingo(f.read()) if solver == "clingo" else extract_conflicts_wasp(f.read()) 
        # print(params, conflits)
        row = {
            "solver": solver,
            "conflicts": conflits
        }
        row.update(params)
        df_input = pd.concat([df_input, pd.DataFrame([row])], ignore_index=True)
    return df_input

df = create_df(df, "clingo")
df = create_df(df, "wasp")
df

,solver,bound,part_size,conflicts,ngroups,alpha,type
0,clingo,60,10,27,10.0,0.60,sat
1,clingo,600,100,232,10.0,0.60,sat
2,clingo,505,10,22,10.0,0.90,unsat
3,clingo,150,100,74,10.0,0.15,sat
4,clingo,9000,1000,98938,10.0,0.90,sat
5,clingo,15,10,0,10.0,0.15,sat
6,clingo,370,10,4845,10.0,0.60,unsat
7,clingo,45550,100,64,10.0,0.90,unsat
8,clingo,30700,100,800411,10.0,0.60,unsat
9,clingo,3007000,1000,74180,10.0,0.60,unsat


In [148]:
table_latex = r"""
\begin{tabular}{rrr r rrr r rrr}
        \toprule
        \multicolumn{3}{c}{$\mathit{part\_size}=10$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=100$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=1000$} \\
        \cmidrule(lr){1-3}
        \cmidrule(lr){4-6}
        \cmidrule(lr){7-9}
        Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp} \\
        \midrule
        15 & XXX         & XXX
        & 150 & XXX        & XXX
        & 1500 & XXX  & XXX \\

        45 & XXX         & XXX
        & 450 & XXX       & XXX
        & 4,500 & XXX  & XXX \\

        60        & XXX & XXX
        & 600 & XXX       & XXX
        & 6,000 & XXX & XXX \\

        90 & XXX & XXX
        & 900 & XXX    & XXX
        & 9,000 & XXX  & XXX \\
        \midrule
        168 & XXX & XXX
        & 8,425 & XXX & XXX
        & 759,250 & XXX & XXX \\

        302 & XXX   & XXX
        & 23,275 & XXX & XXX
        & 2,257,750 & XXX & XXX \\

        370 & XXX     & XXX
        & 30,700 & XXX & XXX
        & 3,007,000 & XXX & XXX \\

        505 & XXX        & XXX
        & 45,550 & XXX      & XXX
        & 4,505,500 & XXX    & XXX \\
        \bottomrule
    \end{tabular}
"""

In [149]:
conflicts_list = []
for id, row in df.iterrows():
    t = (row["type"], row["alpha"], row["bound"], row["solver"], row["conflicts"])
    conflicts_list.append(t)

conflicts_list.sort()
conflicts_list

for item in conflicts_list:
    table_latex = table_latex.replace("XXX", f"{int(item[4]):,}", count=1)
print(table_latex)


\begin{tabular}{rrr r rrr r rrr}
        \toprule
        \multicolumn{3}{c}{$\mathit{part\_size}=10$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=100$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=1000$} \\
        \cmidrule(lr){1-3}
        \cmidrule(lr){4-6}
        \cmidrule(lr){7-9}
        Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp} \\
        \midrule
        15 & 0         & 0
        & 150 & 74        & 549
        & 1500 & 6  & 6,397 \\

        45 & 0         & 1,506
        & 450 & 124       & 102,882
        & 4,500 & 3,004  & 9,182 \\

        60        & 27 & 70,594
        & 600 & 232       & 96,646
        & 6,000 & 4,112 & 8,506 \\

        90 & 5,097,979 & 220,209
        & 900 & 47,209    & 71,775
        & 9,000 & 98,938  & 7,672 \\
        \midrule
        168 & 5,929,256 & 240,305
        & 8,425 & 483,020 & 54,279
        & 759,250 & 113,820 & 11,253 \\

 

In [150]:
print(table_latex.replace("XXX", "CIAO",count=3))


\begin{tabular}{rrr r rrr r rrr}
        \toprule
        \multicolumn{3}{c}{$\mathit{part\_size}=10$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=100$} &
        \multicolumn{3}{c}{$\mathit{part\_size}=1000$} \\
        \cmidrule(lr){1-3}
        \cmidrule(lr){4-6}
        \cmidrule(lr){7-9}
        Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp}
        & Bound & \textsc{clingo} & \textsc{wasp} \\
        \midrule
        15 & 0         & 0
        & 150 & 74        & 549
        & 1500 & 6  & 6,397 \\

        45 & 0         & 1,506
        & 450 & 124       & 102,882
        & 4,500 & 3,004  & 9,182 \\

        60        & 27 & 70,594
        & 600 & 232       & 96,646
        & 6,000 & 4,112 & 8,506 \\

        90 & 5,097,979 & 220,209
        & 900 & 47,209    & 71,775
        & 9,000 & 98,938  & 7,672 \\
        \midrule
        168 & 5,929,256 & 240,305
        & 8,425 & 483,020 & 54,279
        & 759,250 & 113,820 & 11,253 \\

 